# SSL, Fine Tuning, and Linear Probing Heads

In [ ]:
#| default_exp heads

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
from torch import nn

## Linear Probing and Fine Tuning Heads

In [ ]:
#| export
class AvgPatchLogisticRegression(nn.Module):
    """
    Binary logistic regression with progressive dimension reduction
    """
    def __init__(self, 
                 c_in=7,           
                 input_size=512,   
                 dropout=0.
                 ):
        super().__init__()
        self.c_in = c_in
        self.input_size = input_size 
        self.dropout = dropout
        
        # Then reduce across patches using average pooling
        self.pool_patches = nn.AdaptiveAvgPool1d(output_size=1)
                
        # Final classification layers
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(c_in * input_size, 2)
        
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, return_softmax=False):
        """
        Args:
             in: [bs x n channels x d model x n patches]
            return_softmax: If True, applies sigmoid activation
        Returns:
            Binary predictions [bs x 2 x 1]
        """
        bs = x.size(0)
        
        # Reduce embedding dimension first
        x = x.reshape(bs * self.c_in, self.input_size, -1)  # Combine batch and channel dims [bs*c_in x input_size x n_patches]
        # Reduce patch dimension
        x = self.pool_patches(x)  # [bs*7 x input_size x 1]
        # Reshape and flatten for final classification
        x = x.reshape(bs, self.c_in * self.input_size)  # [bs x (c in*input size)]
        # Final classification
        x = self.dropout(x)
        x = self.classifier(x) # [bs x 2]
        
        if return_softmax:
            x = self.sigmoid(x)
            
        return x.unsqueeze(-1)  # [bs x 2 x 1]

In [ ]:
#| notest
m = AvgPatchLogisticRegression(c_in=7, 
                      input_size = 512, 
                      )

x = torch.randn((4,7,512,3600))
y = torch.randint(0, 2, (4,1))
out = m(x, return_softmax=False)
out.shape


torch.Size([4, 2, 1])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()